## Osteoarthritis (Fu et al., 2025)
**Reference**: Fu et al. "Constructing machine learning-based risk prediction model for 
osteoarthritis in population aged 45 and above: NHANES 2011-2018." Scientific Reports, 2025.

**Label**: Self-reported arthritis (MCQ160A) AND osteoarthritis specifically (MCQ195=1)

**Inclusion criteria**: Adults ≥45 years

**Exclusions**: Rheumatoid arthritis (MCQ195=2/3), cancer (MCQ220=1), fibromyalgia (MCQ160N=1)

**Deviations from reference study**:
- 5-fold stratified CV instead of train/test split

In [1]:
import pandas as pd
import numpy as np
import os, pickle

full = pd.read_pickle("/Users/alva/Documents/nhanes_tasks/data/NHANES/NHANES_master.pkl")

In [5]:
# --- Osteoarthritis task ---
# Reference: Fu et al. (2025), Scientific Reports
# Population: Adults ≥45, NHANES 2011-2018
# Label: Self-reported arthritis (MCQ160A) AND osteoarthritis specifically (MCQ195==1)
# Exclusions: rheumatoid arthritis (MCQ195 2/3), cancer (MCQ220), fibromyalgia (MCQ160N)
# Deviation: All cycles except 2017-2020 included; 5-fold stratified CV

subset_oa = full[
    (full['RIDAGEYR'] >= 45) &
    (full['YEAR'].isin(['2011-2012', '2013-2014', '2015-2016', '2017-2018']))
].copy()

subset_oa['OA'] = (
    (subset_oa['MCQ160A'] == 1) &
    (subset_oa['MCQ195'] == 1)
).astype(int)

subset_oa = subset_oa[~(subset_oa['MCQ195'].isin([2, 3]))]
subset_oa = subset_oa[~(subset_oa['MCQ220'] == 1)]
subset_oa = subset_oa[~(subset_oa['MCQ160N'] == 1)]

features_oa = [
    'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDEDUC2', 'SMQ020',
    'BPXDI1', 'BMXBMI', 'BMXWAIST',
    'LBXSCH', 'LBXSTR', 'LBXSPH', 'LBXSTP',
    'LBXNEPCT', 'LBXLYPCT', 'LBXBAPCT', 'LBXSGL', 'LBXSUA', 'LBDHDD'
]

data_oa = subset_oa[features_oa + ['OA']].dropna()
X_oa = data_oa[features_oa].values
y_oa = data_oa['OA'].values

print(f"Osteoarthritis: n={len(y_oa)}, prevalence={y_oa.mean():.3f}, features={X_oa.shape[1]}")

Osteoarthritis: n=7704, prevalence=0.166, features=18


In [6]:
os.makedirs('/Users/alva/Documents/nhanes_tasks/data/tasks/', exist_ok=True)

task = {
    'X': X_oa,
    'y': y_oa,
    'features': features_oa,
    'name': 'osteoarthritis'
}

with open('/Users/alva/Documents/nhanes_tasks/data/tasks/osteoarthritis.pkl', 'wb') as f:
    pickle.dump(task, f)

print("Saved.")

Saved.
